In [1]:
import pandas as pd
import gc
import numpy as np

In [2]:
df3 = pd.read_pickle('../data/df3.pkl')

In [3]:
df3.info()
df3.shape # (7483231, 30)
#df3.head(20)

<class 'pandas.DataFrame'>
RangeIndex: 7483231 entries, 0 to 7483230
Data columns (total 30 columns):
 #   Column              Dtype         
---  ------              -----         
 0   BUS_SVC_NUM         float64       
 1   CRD_NUM             object        
 2   DEST_LOC_ID_NUM     float64       
 3   ENTRY_DT            datetime64[us]
 4   ENTRY_TM            datetime64[us]
 5   EXIT_DT             datetime64[us]
 6   EXIT_TM             datetime64[us]
 7   JRNY_ID_NUM         object        
 8   ORIG_LOC_ID_NUM     float64       
 9   RIDE_DISC_AMT       float64       
 10  RIDE_DIST_KM_CNT    float64       
 11  RIDE_FARE_AMT       float64       
 12  RIDE_ID_NUM         object        
 13  RIDE_MIN_CNT        float64       
 14  PATRON_CATG_ID_NUM  int64         
 15  TRNSPT_MODE_CD      int64         
 16  DEST_STATION_NAME   str           
 17  DEST_MRK_ID_NUM     float64       
 18  DEST_LATITUDE       float64       
 19  DEST_LONGITUDE      float64       
 20  DEST_Travel_T

(7483231, 30)

### Information on the Patron Categories: 

- Student (Primary 1 to JC/Poly) --> 7 - 19 years old
- Adult --> 20 - 59 years old
- Senior Citizen --> 60 years and above

Average Walking Speeds by Age:
- Children (6-12 years): Around 3.0 to 4.5 km/h (1.9 to 2.8 mph)
- Adolescents (13-19 years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)
- Adults (20-59 years): Around 5.0 to 6.0 km/h (3.1 to 3.7 mph)
- Older Adults (60+ years): Around 4.0 to 5.0 km/h (2.5 to 3.1 mph)

To calculate the walking speed of each group, I take the midpoint of the range. 
For Children and Adolescents, I took the midpoint of both ranges and took the average.

In [4]:
# We first filter for 3 relevant PATRON_CATG_ID_NUM.
patron_map = {1: 'Adult', 3: 'Student', 4: 'Senior Citizen'}
df3['PATRON_CATG_DESC_TXT'] = df3['PATRON_CATG_ID_NUM'].map(patron_map)

walking_speed_ms = {
    'Student': 4.125 / 3.6,      # (3.75 + 4.5) / 2 = 4.125 km/h → ~1.146 m/s
    'Adult': 1.08,          # 1.08 m/s
    'Senior Citizen': 0.80  # 1.25 m/s
}

df3['walking_speed_ms'] = df3['PATRON_CATG_DESC_TXT'].map(walking_speed_ms)

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: df3 is already sorted by CRD_NUM and ENTRY_TM. Flag last row of each CRD_NUM
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [5]:
df3['service_day'] = (df3['ENTRY_TM'] - pd.Timedelta(hours=5)).dt.date
target_day = pd.Timestamp('2025-02-12').date()

df3 = df3[df3['service_day'] == target_day].reset_index(drop=True)

In [6]:
df3 = df3.sort_values(["CRD_NUM", "ENTRY_TM"]).reset_index(drop=True)

In [7]:
df3["next_ENTRY_TM"] = df3.groupby("CRD_NUM")["ENTRY_TM"].shift(-1)
df3["next_ORIG_LOC_ID_NUM"] = df3.groupby("CRD_NUM")["ORIG_LOC_ID_NUM"].shift(-1)
df3["next_BUS_SVC_NUM"] = df3.groupby("CRD_NUM")["BUS_SVC_NUM"].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)

In [8]:
df3.shape

(7468612, 37)

In [9]:
df3["is_last_stage"] = df3["next_ENTRY_TM"].isna()

df3['missing_info'] = (
    df3['ENTRY_TM'].isna() |
    df3['EXIT_TM'].isna() |
    df3['ORIG_LATITUDE'].isna() |
    df3['ORIG_LONGITUDE'].isna() |
    df3['DEST_LATITUDE'].isna() |
    df3['DEST_LONGITUDE'].isna() |
    df3["ORIG_LOC_ID_NUM"].isna() |
    df3["DEST_LOC_ID_NUM"].isna()
)

In [10]:
#create next bus and next station columns by shifting
# flags consecutive rides on the same bus service number/same station
df3["same_bus_service"] = (
    df3["BUS_SVC_NUM"].notna() &
    df3["next_BUS_SVC_NUM"].notna() &
    (df3["BUS_SVC_NUM"] == df3["next_BUS_SVC_NUM"])
)
df3["same_station_consecutive"] = (
    df3["DEST_STATION_NAME"].notna() &
    df3["next_orig_station"].notna() &
    (df3["DEST_STATION_NAME"] == df3["next_orig_station"])
)

In [11]:
df3["return_or_intermediate"] = (
    df3["same_bus_service"] |
    df3["same_station_consecutive"]
)

In [12]:
df3["journey_termination_flag"] = (
    df3["is_last_stage"] |
    df3["missing_info"] |
    df3["return_or_intermediate"]
)

In [13]:
df3['journey_termination_flag'].value_counts()


journey_termination_flag
True     4483070
False    2985542
Name: count, dtype: int64

In [14]:
# Summary counts
print("Unique commuters:", df3["CRD_NUM"].nunique())
print("Last stages:", df3["is_last_stage"].sum())

print("\nMissing info:")
print(df3["missing_info"].value_counts(dropna=False))

print("\nSame bus service:")
print(df3["same_bus_service"].value_counts(dropna=False))

print("\nSame station consecutive:")
print(df3["same_station_consecutive"].value_counts(dropna=False))

print("\nReturn / intermediate:")
print(df3["return_or_intermediate"].value_counts(dropna=False))

print("\nJourney termination flag:")
print(df3["journey_termination_flag"].value_counts(dropna=False))

Unique commuters: 2453068
Last stages: 2453068

Missing info:
missing_info
False    7400602
True       68010
Name: count, dtype: int64

Same bus service:
same_bus_service
False    6918191
True      550421
Name: count, dtype: int64

Same station consecutive:
same_station_consecutive
False    5937279
True     1531333
Name: count, dtype: int64

Return / intermediate:
return_or_intermediate
False    5472581
True     1996031
Name: count, dtype: int64

Journey termination flag:
journey_termination_flag
True     4483070
False    2985542
Name: count, dtype: int64


In [15]:
# Reasons why flagged.
df3["journey_termination_reason"] = np.select(
    [
        df3["is_last_stage"],
        df3["missing_info"],
        df3["return_or_intermediate"]
    ],
    [
        "last_stage",
        "missing_info",
        "return_or_intermediate"
    ],
    default="candidate_transfer"
)

In [16]:
print(df3["journey_termination_reason"].value_counts(dropna=False))

journey_termination_reason
candidate_transfer        2985542
last_stage                2453068
return_or_intermediate    1984550
missing_info                45452
Name: count, dtype: int64


In [17]:
df4 = df3.copy()

#define full journey sequence
df4['full_journey_seq'] = (
    df4.groupby('CRD_NUM')['journey_termination_flag']
       .shift(fill_value=False)
       .groupby(df4['CRD_NUM'])
       .cumsum()
)

#first row of each full journey
journey_first = (
    df4.drop_duplicates(subset=['CRD_NUM', 'full_journey_seq'], keep='first')
       [['CRD_NUM', 'full_journey_seq', 'ORIG_STATION_NAME', 'ENTRY_TM']]
       .rename(columns={
           'ORIG_STATION_NAME': 'orig_journey_station',
           'ENTRY_TM': 'entry_journey_tm'
       })
)

#last row of each full journey
journey_last = (
    df4.drop_duplicates(subset=['CRD_NUM', 'full_journey_seq'], keep='last')
       [['CRD_NUM', 'full_journey_seq',
         'DEST_STATION_NAME', 'EXIT_TM',
         'next_orig_station', 'walk_distance']]
       .rename(columns={
           'DEST_STATION_NAME': 'dest_journey_station',
           'EXIT_TM': 'exit_journey_tm',
           'next_orig_station': 'next_orig_journey_location',
           'walk_distance': 'walk_to_next_journey_distance'
       })
)

#combine first + last journey info
journey_info = journey_first.merge(
    journey_last,
    on=['CRD_NUM', 'full_journey_seq'],
    how='inner'
)

#merge back to every row
df4 = df4.merge(
    journey_info,
    on=['CRD_NUM', 'full_journey_seq'],
    how='left'
)


#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
    - Allowance = Walking speed * walking distance



Describe.

In [18]:
TEMPORAL_SPEC = "lenient"

In [19]:
# Calculate time gap mins

# Shift next entry time within the same commuter group
df3['next_ENTRY_TM'] = df3.groupby('CRD_NUM')['ENTRY_TM'].shift(-1)
df3["next_TRNSPT_MODE_CD"] = df3.groupby("CRD_NUM")["TRNSPT_MODE_CD"].shift(-1)

In [20]:
df3["is_peak"] = (
    (
        ((df3["next_ENTRY_TM"].dt.hour == 6) & (df3["next_ENTRY_TM"].dt.minute >= 30)) |
        (df3["next_ENTRY_TM"].dt.hour.isin([7, 8]))
    ) |
    (df3["next_ENTRY_TM"].dt.hour.isin([17, 18, 19]))
)

df3["peak_period"] = np.select(
    [
        (
            ((df3["next_ENTRY_TM"].dt.hour == 6) & (df3["next_ENTRY_TM"].dt.minute >= 30)) |
            (df3["next_ENTRY_TM"].dt.hour.isin([7, 8]))
        ),
        (df3["next_ENTRY_TM"].dt.hour.isin([17, 18, 19]))
    ],
    [
        "morning_peak",
        "evening_peak"
    ],
    default="off_peak"
)


In [21]:
df3["mode_pair"] = np.select(
    [
        (df3["TRNSPT_MODE_CD"] == 1) & (df3["next_TRNSPT_MODE_CD"] == 1),
        (df3["TRNSPT_MODE_CD"] == 2) & (df3["next_TRNSPT_MODE_CD"] == 1),
        (df3["TRNSPT_MODE_CD"] == 1) & (df3["next_TRNSPT_MODE_CD"] == 2),
        (df3["TRNSPT_MODE_CD"] == 2) & (df3["next_TRNSPT_MODE_CD"] == 2)
    ],
    [
        "bus_bus",
        "train_bus",
        "bus_train",
        "train_train"
    ],
    default="other"
)

In [22]:
df3["time_gap_mins"] = (
    df3["next_ENTRY_TM"] - df3["EXIT_TM"]
).dt.total_seconds() / 60

df3["predicted_walking_time_mins"] = (
    df3["walk_distance"] / df3["walking_speed_ms"]
) / 60

df3["waiting_time_plus_extra"] = (
    df3["time_gap_mins"] - df3["predicted_walking_time_mins"]
)

In [23]:
if TEMPORAL_SPEC == "strict":
    df3["waiting_time_allowed"] = np.select(
        [
            (df3["mode_pair"] == "bus_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "bus_train") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_train") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_train") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_train") & (~df3["is_peak"])
        ],
        [
            8,   # bus_bus peak
            12,  # bus_bus off_peak
            8,   # train_bus peak
            12,  # train_bus off_peak
            6,   # bus_train peak
            8,   # bus_train off_peak
            6,   # train_train peak
            8    # train_train off_peak
        ],
        default=np.nan
    )

elif TEMPORAL_SPEC == "baseline":
    df3["waiting_time_allowed"] = np.select(
        [
            (df3["mode_pair"] == "bus_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "bus_train") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_train") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_train") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_train") & (~df3["is_peak"])
        ],
        [
            10,  # bus_bus peak
            15,  # bus_bus off_peak
            10,  # train_bus peak
            15,  # train_bus off_peak
            8,   # bus_train peak
            10,  # bus_train off_peak
            8,   # train_train peak
            10   # train_train off_peak
        ],
        default=np.nan
    )

elif TEMPORAL_SPEC == "lenient":
    df3["waiting_time_allowed"] = np.select(
        [
            (df3["mode_pair"] == "bus_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_bus") & (~df3["is_peak"]),
            (df3["mode_pair"] == "bus_train") & (df3["is_peak"]),
            (df3["mode_pair"] == "bus_train") & (~df3["is_peak"]),
            (df3["mode_pair"] == "train_train") & (df3["is_peak"]),
            (df3["mode_pair"] == "train_train") & (~df3["is_peak"])
        ],
        [
            12,  # bus_bus peak
            18,  # bus_bus off_peak
            12,  # train_bus peak
            18,  # train_bus off_peak
            10,  # bus_train peak
            12,  # bus_train off_peak
            10,  # train_train peak
            12   # train_train off_peak
        ],
        default=np.nan
    )

else:
    raise ValueError("TEMPORAL_SPEC must be 'strict', 'baseline', or 'lenient'")

In [24]:
df3["extras_allowed"] = 0
df3["Total_allowance"] = df3["waiting_time_allowed"] + df3["extras_allowed"]

In [25]:
df3["temporal_flag_reason"] = np.select(
    [
        df3["time_gap_mins"].isna(),
        df3["predicted_walking_time_mins"].isna(),
        df3["waiting_time_plus_extra"].isna(),
        df3["waiting_time_allowed"].isna(),
        df3["waiting_time_plus_extra"] > df3["Total_allowance"],
    ],
    [
        "null_time_gap",
        "null_predicted_walking_time",
        "null_waiting_time_plus_extra",
        "unknown_mode_pair",
        "exceeds_total_allowance",
    ],
    default="within_total_allowance"
)

df3["exceeds_time_allowance"] = (
    df3["temporal_flag_reason"] != "within_total_allowance"
)

In [26]:
print(df3["exceeds_time_allowance"].value_counts())
print(f"\nTotal temporal new journey flags: {df3['exceeds_time_allowance'].sum():,}")
print(df3["temporal_flag_reason"].value_counts(dropna=False))

print("\nMode pair:")
print(df3["mode_pair"].value_counts(dropna=False))

print("\nPeak period:")
print(df3["peak_period"].value_counts(dropna=False))

print("\nWaiting time allowed:")
print(df3["waiting_time_allowed"].value_counts(dropna=False))

exceeds_time_allowance
True     5390907
False    2077705
Name: count, dtype: int64

Total temporal new journey flags: 5,390,907
temporal_flag_reason
exceeds_total_allowance        2730151
null_time_gap                  2498375
within_total_allowance         2077705
null_predicted_walking_time     162381
Name: count, dtype: int64

Mode pair:
mode_pair
other          2453068
bus_bus        1829510
train_train    1230987
bus_train       978125
train_bus       976922
Name: count, dtype: int64

Peak period:
peak_period
off_peak        5179462
evening_peak    1672738
morning_peak     616412
Name: count, dtype: int64

Waiting time allowed:
waiting_time_allowed
NaN     2453068
12.0    2371658
18.0    1580584
10.0    1063302
Name: count, dtype: int64


In [27]:
df3["temporal_full_journey_seq"] = (
    df3.groupby("CRD_NUM")["exceeds_time_allowance"]
       .shift(fill_value=False)
       .groupby(df3["CRD_NUM"])
       .cumsum()
)

# first row of each temporal-only journey
temporal_journey_first = (
    df3.drop_duplicates(subset=["CRD_NUM", "temporal_full_journey_seq"], keep="first")
       [["CRD_NUM", "temporal_full_journey_seq", "ORIG_STATION_NAME", "ENTRY_TM"]]
       .rename(columns={
           "ORIG_STATION_NAME": "(TEMPORAL_Only)_orig_journey",
           "ENTRY_TM": "(TEMPORAL_Only)_entry_journey_tm"
       })
)

# last row of each temporal-only journey
temporal_journey_last = (
    df3.drop_duplicates(subset=["CRD_NUM", "temporal_full_journey_seq"], keep="last")
       [["CRD_NUM", "temporal_full_journey_seq", "DEST_STATION_NAME", "EXIT_TM"]]
       .rename(columns={
           "DEST_STATION_NAME": "(TEMPORAL_Only)_dest_journey",
           "EXIT_TM": "(TEMPORAL_Only)_exit_journey_tm"
       })
)

# combine first + last journey info
temporal_journey_info = temporal_journey_first.merge(
    temporal_journey_last,
    on=["CRD_NUM", "temporal_full_journey_seq"],
    how="inner"
)

# merge back to df3
df3 = df3.merge(
    temporal_journey_info,
    on=["CRD_NUM", "temporal_full_journey_seq"],
    how="left"
)

* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

## Using Binary and Temporal
- apply temporal criteria only after filtering transfers from binary

In [28]:
temporal_cols = [
    'CRD_NUM',
    'ENTRY_TM',
    'EXIT_TM',
    'time_gap_mins',
    'predicted_walking_time_mins',
    'waiting_time_plus_extra',
    'next_TRNSPT_MODE_CD',
    'is_peak',
    'peak_period',
    'mode_pair',
    'waiting_time_allowed',
    'extras_allowed',
    'Total_allowance',
    'exceeds_time_allowance',
    'temporal_flag_reason'
]

df4 = df4.merge(
    df3[temporal_cols],
    on=['CRD_NUM', 'ENTRY_TM', 'EXIT_TM'],
    how='left'
)

# combined rule terminate if binary OR temporal says so
df4['final_termination_flag'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance']
)

df4['final_termination_reason'] = ''

# binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

# temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

df4['final_termination_reason'] = (
    df4['final_termination_reason']
    .str.rstrip(' | ')
)

# default
df4.loc[df4['final_termination_reason'] == '', 'final_termination_reason'] = 'candidate_transfer'

print(df4['final_termination_flag'].value_counts(dropna=False))
print(df4['final_termination_reason'].value_counts(dropna=False))

final_termination_flag
True     5826431
False    1677301
Name: count, dtype: int64
final_termination_reason
last_stage | null_time_gap                              2455552
candidate_transfer                                      1677301
return_or_intermediate | exceeds_total_allowance        1518156
exceeds_total_allowance                                 1211957
return_or_intermediate                                   400350
null_predicted_walking_time                               96284
missing_info | null_time_gap                              77943
return_or_intermediate | null_predicted_walking_time      66044
missing_info                                                 54
missing_info | null_predicted_walking_time                   53
missing_info | exceeds_total_allowance                       38
Name: count, dtype: int64


## Hard geographic threshold (consider)

In [29]:
SPATIAL_THRESHOLD_M = 1200

df3['exceeds_walking_dist_threshold'] = (
    df3['walk_distance'] > SPATIAL_THRESHOLD_M
)

df3['spatial_flag_reason'] = np.select(
    [
        df3['walk_distance'].isna(),
        df3['walk_distance'] > SPATIAL_THRESHOLD_M
    ],
    [
        'null_walk_distance',
        'exceeds_distance_threshold'
    ],
    default='within_distance_threshold'
)

print(df3['exceeds_walking_dist_threshold'].value_counts(dropna=False))
print(df3['spatial_flag_reason'].value_counts(dropna=False))

exceeds_walking_dist_threshold
False    7241541
True      227071
Name: count, dtype: int64
spatial_flag_reason
within_distance_threshold     4757833
null_walk_distance            2483708
exceeds_distance_threshold     227071
Name: count, dtype: int64


In [30]:
df3['spatial_full_journey_seq'] = (
    df3.groupby('CRD_NUM')['exceeds_walking_dist_threshold']
       .shift(fill_value=False)
       .groupby(df3['CRD_NUM'])
       .cumsum()
)

#first row of each spatial-only full journey
spatial_journey_first = (
    df3.drop_duplicates(subset=['CRD_NUM', 'spatial_full_journey_seq'], keep='first')
       [['CRD_NUM', 'spatial_full_journey_seq', 'ORIG_STATION_NAME']]
       .rename(columns={
           'ORIG_STATION_NAME': '(SPATIAL_Only)_orig_journey'
       })
)

#last row of each spatial-only full journey
spatial_journey_last = (
    df3.drop_duplicates(subset=['CRD_NUM', 'spatial_full_journey_seq'], keep='last')
       [['CRD_NUM', 'spatial_full_journey_seq', 'DEST_STATION_NAME']]
       .rename(columns={
           'DEST_STATION_NAME': '(SPATIAL_Only)_dest_journey'
       })
)

#combine first + last journey info
spatial_journey_info = spatial_journey_first.merge(
    spatial_journey_last,
    on=['CRD_NUM', 'spatial_full_journey_seq'],
    how='inner'
)

#merge back to every row in df3
df3 = df3.merge(
    spatial_journey_info,
    on=['CRD_NUM', 'spatial_full_journey_seq'],
    how='left'
)

In [31]:
df3['(SPATIAL_Only)_short_round_trip_flag'] = (
    (df3['(SPATIAL_Only)_orig_journey'] == df3['(SPATIAL_Only)_dest_journey']) &
    df3['(SPATIAL_Only)_orig_journey'].notna() &
    df3['(SPATIAL_Only)_dest_journey'].notna()
)

df3['(SPATIAL_Only)_round_trip_reason'] = np.select(
    [
        df3['(SPATIAL_Only)_orig_journey'].isna() |
        df3['(SPATIAL_Only)_dest_journey'].isna(),

        df3['(SPATIAL_Only)_orig_journey'] == df3['(SPATIAL_Only)_dest_journey']
    ],
    [
        'null_origin_or_destination',
        'same_origin_and_destination'
    ],
    default='different_origin_destination'
)

print(df3['(SPATIAL_Only)_short_round_trip_flag'].value_counts(dropna=False))
print(df3['(SPATIAL_Only)_round_trip_reason'].value_counts(dropna=False))

(SPATIAL_Only)_short_round_trip_flag
False    5711520
True     1757092
Name: count, dtype: int64
(SPATIAL_Only)_round_trip_reason
different_origin_destination    5681444
same_origin_and_destination     1757092
null_origin_or_destination        30076
Name: count, dtype: int64


In [32]:
df3['spatial_break_flag_final'] = (
    df3['exceeds_walking_dist_threshold'] |
    df3['(SPATIAL_Only)_short_round_trip_flag']
)

print(df3['spatial_break_flag_final'].value_counts(dropna=False))

spatial_break_flag_final
False    5488912
True     1979700
Name: count, dtype: int64


### Combining all 3 criteria

In [33]:
spatial_cols = [
    'CRD_NUM',
    'ENTRY_TM',
    'EXIT_TM',
    'exceeds_walking_dist_threshold',
    'spatial_flag_reason',
    'spatial_break_flag_final'
]

df4 = df4.merge(
    df3[spatial_cols],
    on=['CRD_NUM', 'ENTRY_TM', 'EXIT_TM'],
    how='left'
)

#bin + temp
df4['final_termination_flag'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance']
)

df4['final_termination_reason'] = ''

# binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

# temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

# clean trailing separator
df4['final_termination_reason'] = (
    df4['final_termination_reason']
    .str.rstrip(' | ')
)

# default
df4.loc[df4['final_termination_reason'] == '', 'final_termination_reason'] = 'candidate_transfer'

print(df4['final_termination_flag'].value_counts(dropna=False))
print(df4['final_termination_reason'].value_counts(dropna=False))


final_termination_flag
True     6002031
False    1677301
Name: count, dtype: int64
final_termination_reason
last_stage | null_time_gap                              2467972
candidate_transfer                                      1677301
return_or_intermediate | exceeds_total_allowance        1518156
exceeds_total_allowance                                 1211957
return_or_intermediate                                   400350
missing_info | null_time_gap                             241123
null_predicted_walking_time                               96284
return_or_intermediate | null_predicted_walking_time      66044
missing_info                                                 54
missing_info | null_predicted_walking_time                   53
missing_info | exceeds_total_allowance                       38
Name: count, dtype: int64


In [34]:
df4['final_full_journey_seq'] = (
    df4.groupby('CRD_NUM')['final_termination_flag']
       .shift(fill_value=False)
       .groupby(df4['CRD_NUM'])
       .cumsum()
)

# first row of each binary + temporal journey
final_journey_first = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_full_journey_seq'], keep='first')
       [['CRD_NUM', 'final_full_journey_seq', 'ORIG_STATION_NAME']]
       .rename(columns={
           'ORIG_STATION_NAME': 'final_orig_journey_station'
       })
)

# last row of each binary + temporal journey
final_journey_last = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_full_journey_seq'], keep='last')
       [['CRD_NUM', 'final_full_journey_seq', 'DEST_STATION_NAME']]
       .rename(columns={
           'DEST_STATION_NAME': 'final_dest_journey_station'
       })
)

# combine first + last journey info
final_journey_info = final_journey_first.merge(
    final_journey_last,
    on=['CRD_NUM', 'final_full_journey_seq'],
    how='inner'
)

# merge back to df4
df4 = df4.merge(
    final_journey_info,
    on=['CRD_NUM', 'final_full_journey_seq'],
    how='left'
)

In [35]:
#now round trip is based on binary + temporal journeys, not binary-only journeys
df4['short_round_trip_flag'] = (
    (df4['final_orig_journey_station'] == df4['final_dest_journey_station']) &
    df4['final_orig_journey_station'].notna() &
    df4['final_dest_journey_station'].notna()
)

df4['round_trip_reason'] = np.select(
    [
        df4['final_orig_journey_station'].isna() |
        df4['final_dest_journey_station'].isna(),

        df4['final_orig_journey_station'] == df4['final_dest_journey_station']
    ],
    [
        'null_origin_or_destination',
        'same_origin_and_destination'
    ],
    default='different_origin_destination'
)

print(df4['short_round_trip_flag'].value_counts(dropna=False))
print(df4['round_trip_reason'].value_counts(dropna=False))


short_round_trip_flag
False    7661388
True       17944
Name: count, dtype: int64
round_trip_reason
different_origin_destination    7626451
null_origin_or_destination        34937
same_origin_and_destination       17944
Name: count, dtype: int64


In [36]:
df4['final_termination_flag_spatial'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance'] |
    df4['exceeds_walking_dist_threshold'] |
    df4['short_round_trip_flag']
)

df4['final_termination_reason_spatial'] = ''

# binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason_spatial'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

# temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason_spatial'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

# spatial 1: walking distance too far
df4.loc[df4['exceeds_walking_dist_threshold'], 'final_termination_reason_spatial'] += (
    df4['spatial_flag_reason'].fillna('') + ' | '
)

# spatial 2: short round trip
df4.loc[df4['short_round_trip_flag'], 'final_termination_reason_spatial'] += (
    df4['round_trip_reason'].fillna('') + ' | '
)

# clean trailing separator
df4['final_termination_reason_spatial'] = (
    df4['final_termination_reason_spatial']
    .str.rstrip(' | ')
)

# default
df4.loc[df4['final_termination_reason_spatial'] == '', 'final_termination_reason_spatial'] = 'candidate_transfer'

print(df4['final_termination_flag_spatial'].value_counts(dropna=False))
print(df4['final_termination_reason_spatial'].value_counts(dropna=False))

final_termination_flag_spatial
True     6026404
False    1652928
Name: count, dtype: int64
final_termination_reason_spatial
last_stage | null_time_gap                                                                                     2450913
candidate_transfer                                                                                             1652928
return_or_intermediate | exceeds_total_allowance                                                               1506894
exceeds_total_allowance                                                                                        1037860
return_or_intermediate                                                                                          396337
missing_info | null_time_gap | exceeds_distance_threshold                                                       192476
exceeds_total_allowance | exceeds_distance_threshold                                                            172772
null_predicted_walking_time                

In [37]:
df4['final_termination_flag_spatial_nodist'] = (
    df4['journey_termination_flag'] |
    df4['exceeds_time_allowance'] |
    df4['short_round_trip_flag']
)

df4['final_termination_reason_spatial_nodist'] = ''

# binary
df4.loc[df4['journey_termination_flag'], 'final_termination_reason_spatial_nodist'] += (
    df4['journey_termination_reason'].fillna('') + ' | '
)

# temporal
df4.loc[df4['exceeds_time_allowance'], 'final_termination_reason_spatial_nodist'] += (
    df4['temporal_flag_reason'].fillna('') + ' | '
)

# spatial (round trip only)
df4.loc[df4['short_round_trip_flag'], 'final_termination_reason_spatial_nodist'] += (
    df4['round_trip_reason'].fillna('') + ' | '
)

# clean trailing separator
df4['final_termination_reason_spatial_nodist'] = (
    df4['final_termination_reason_spatial_nodist']
    .str.rstrip(' | ')
)

# default
df4.loc[
    df4['final_termination_reason_spatial_nodist'] == '',
    'final_termination_reason_spatial_nodist'
] = 'candidate_transfer'

print(df4['final_termination_flag_spatial_nodist'].value_counts(dropna=False))
print(df4['final_termination_reason_spatial_nodist'].value_counts(dropna=False))

final_termination_flag_spatial_nodist
True     6006378
False    1672954
Name: count, dtype: int64
final_termination_reason_spatial_nodist
last_stage | null_time_gap                                                            2463313
candidate_transfer                                                                    1672954
return_or_intermediate | exceeds_total_allowance                                      1514523
exceeds_total_allowance                                                               1210632
return_or_intermediate                                                                 396866
missing_info | null_time_gap                                                           240970
null_predicted_walking_time                                                             96153
return_or_intermediate | null_predicted_walking_time                                    65832
last_stage | null_time_gap | same_origin_and_destination                                 4659
same_origin_and_

In [38]:
# Main final classifier: binary + temporal + spatial distance + round trip
df4['is_same_journey_final'] = ~df4['final_termination_flag_spatial']

# Alternative final classifier: binary + temporal + round trip only
df4['is_same_journey_final_nodist'] = ~df4['final_termination_flag_spatial_nodist']

print(df4['is_same_journey_final'].value_counts(dropna=False))
print(df4['is_same_journey_final_nodist'].value_counts(dropna=False))

is_same_journey_final
False    6026404
True     1652928
Name: count, dtype: int64
is_same_journey_final_nodist
False    6006378
True     1672954
Name: count, dtype: int64


In [39]:
import gc
gc.collect()

0

## Internal Validity Check with pt_jrny

In [40]:
df5 = pd.read_pickle('../data/df5.pkl')
df5.info()

<class 'pandas.DataFrame'>
RangeIndex: 5336605 entries, 0 to 5336604
Data columns (total 23 columns):
 #   Column              Dtype         
---  ------              -----         
 0   CRD_NUM             object        
 1   JRNY_DEST_ID_NUM    float64       
 2   JRNY_START_DT       datetime64[us]
 3   JRNY_START_TM       datetime64[us]
 4   JRNY_END_DT         datetime64[us]
 5   JRNY_END_TM         datetime64[us]
 6   JRNY_ORIG_ID_NUM    float64       
 7   JRNY_DIST_KM_CNT    float64       
 8   JRNY_FARE_AMT       float64       
 9   JRNY_ID_NUM         object        
 10  JRNY_TM_MIN_CNT     float64       
 11  PATRON_CATG_ID_NUM  float64       
 12  TRNSPT_MODE_CD      int64         
 13  DEST_STATION_NAME   str           
 14  DEST_MRK_ID_NUM     float64       
 15  DEST_LATITUDE       float64       
 16  DEST_LONGITUDE      float64       
 17  DEST_Travel_Type    str           
 18  ORIG_STATION_NAME   str           
 19  ORIG_MRK_ID_NUM     float64       
 20  ORIG_LATITUDE

In [41]:
df5['service_day'] = (df5['JRNY_START_TM'] - pd.Timedelta(hours=5)).dt.date

target_day = pd.Timestamp('2025-02-12').date()

df5 = df5[df5['service_day'] == target_day].reset_index(drop=True)

In [42]:
# same filtering as df3
df5 = df5[df5['PATRON_CATG_ID_NUM'].isin([1, 3, 4])].reset_index(drop=True)

In [43]:
# Merge on CRD_NUM + JRNY_ID_NUM to attach journey boundaries to each ride
df_val = df4.merge(
    df5[['CRD_NUM', 'JRNY_ID_NUM', 'JRNY_START_TM', 'JRNY_END_TM']],
    on=['CRD_NUM', 'JRNY_ID_NUM'],
    how='inner'
)

# Keep only rides that fall within their journey's time window
df_val = df_val[
    (df_val['ENTRY_TM'] >= df_val['JRNY_START_TM']) &
    (df_val['EXIT_TM'] <= df_val['JRNY_END_TM'])
].reset_index(drop=True)

print(f"Rows after merge + time filter: {len(df_val):,}")

Rows after merge + time filter: 7,198,635


In [44]:
df_val = df_val.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)
# Shift JRNY_ID_NUM within each commuter to get the next ride's journey ID
df_val['next_JRNY_ID_NUM'] = df_val.groupby('CRD_NUM')['JRNY_ID_NUM'].shift(-1)

# true_transfer = True → same journey (actual transfer)
# true_transfer = False → different journey (actual new journey)
df_val['true_transfer'] = (df_val['JRNY_ID_NUM'] == df_val['next_JRNY_ID_NUM'])

# Drop last stage of each commuter — no next ride to compare against
df_val = df_val[df_val['next_JRNY_ID_NUM'].notna()].reset_index(drop=True)

print(df_val['true_transfer'].value_counts())

true_transfer
False    2605293
True     2224718
Name: count, dtype: int64


In [45]:
print("\nMode pair:")
print(df_val['mode_pair'].value_counts(dropna=False))

print("\nPeak period:")
print(df_val['peak_period'].value_counts(dropna=False))

print("\nWaiting time allowed:")
print(df_val['waiting_time_allowed'].value_counts(dropna=False))

print("\nMode pair x peak period:")
print(pd.crosstab(df_val['mode_pair'], df_val['peak_period']))

print("\nMode pair x waiting time allowed:")
print(pd.crosstab(df_val['mode_pair'], df_val['waiting_time_allowed']))


Mode pair:
mode_pair
bus_bus        1733658
train_train    1198513
train_bus       956494
bus_train       940706
other              640
Name: count, dtype: int64

Peak period:
peak_period
off_peak        2619511
evening_peak    1614041
morning_peak     596459
Name: count, dtype: int64

Waiting time allowed:
waiting_time_allowed
12.0    2283765
18.0    1512629
10.0    1032977
NaN         640
Name: count, dtype: int64

Mode pair x peak period:
peak_period  evening_peak  morning_peak  off_peak
mode_pair                                        
bus_bus            543815        168088   1021755
bus_train          248658        245442    446606
other                   0             0       640
train_bus          311556        154064    490874
train_train        510012         28865    659636

Mode pair x waiting time allowed:
waiting_time_allowed    10.0    12.0     18.0
mode_pair                                    
bus_bus                    0  711903  1021755
bus_train             494100  

In [46]:
# helper function
def print_metrics(df, pred_flag_col, label):
    """
    pred_flag_col: True = classifier says NEW JOURNEY, False = classifier says TRANSFER
    true_transfer: True = actual transfer, False = actual new journey
    """
    actual_transfer = df['true_transfer']
    pred_transfer = ~df[pred_flag_col]      # flip: flag=True means new journey, so ~flag = predicted transfer

    tp = (actual_transfer & pred_transfer).sum()     # actual transfer, predicted transfer
    tn = (~actual_transfer & ~pred_transfer).sum()   # actual new journey, predicted new journey
    fp = (~actual_transfer & pred_transfer).sum()    # actual new journey, predicted transfer (merge error)
    fn = (actual_transfer & ~pred_transfer).sum()    # actual transfer, predicted new journey (split error)

    total_actual_transfers = actual_transfer.sum()

    print(f"\n=== {label} ===")
    print(f"TP: {tp:,}  TN: {tn:,}  FP: {fp:,}  FN: {fn:,}")
    print(f"Split rate  (FN / actual transfers): {fn / total_actual_transfers:.4f}")
    print(f"Merge rate  (FP / actual transfers): {fp / total_actual_transfers:.4f}")
    print(f"Sensitivity (TP / (TP+FN)):          {tp / (tp + fn):.4f}")
    print(f"Specificity (TN / (TN+FP)):          {tn / (tn + fp):.4f}")
    print(f"Accuracy    ((TP+TN) / total):       {(tp + tn) / len(df):.4f}")

    print(pd.crosstab(
        actual_transfer,
        pred_transfer,
        rownames=['Actual transfer'],
        colnames=[f'Predicted transfer ({label})']
    ))

In [47]:
print_metrics(df_val, 'journey_termination_flag', 'Binary Criteria')


=== Binary Criteria ===
TP: 1,850,511  TN: 1,549,318  FP: 1,055,975  FN: 374,207
Split rate  (FN / actual transfers): 0.1682
Merge rate  (FP / actual transfers): 0.4747
Sensitivity (TP / (TP+FN)):          0.8318
Specificity (TN / (TN+FP)):          0.5947
Accuracy    ((TP+TN) / total):       0.7039
Predicted transfer (Binary Criteria)    False    True 
Actual transfer                                       
False                                 1549318  1055975
True                                   374207  1850511


In [48]:
print_metrics(df_val, 'exceeds_time_allowance', 'Temporal Criteria')


=== Temporal Criteria ===
TP: 1,963,626  TN: 2,475,666  FP: 129,627  FN: 261,092
Split rate  (FN / actual transfers): 0.1174
Merge rate  (FP / actual transfers): 0.0583
Sensitivity (TP / (TP+FN)):          0.8826
Specificity (TN / (TN+FP)):          0.9502
Accuracy    ((TP+TN) / total):       0.9191
Predicted transfer (Temporal Criteria)    False    True 
Actual transfer                                         
False                                   2475666   129627
True                                     261092  1963626


In [49]:
print_metrics(df_val, 'exceeds_walking_dist_threshold', 'Spatial Criteria (Distance Threshold)')


=== Spatial Criteria (Distance Threshold) ===
TP: 2,215,259  TN: 192,402  FP: 2,412,891  FN: 9,459
Split rate  (FN / actual transfers): 0.0043
Merge rate  (FP / actual transfers): 1.0846
Sensitivity (TP / (TP+FN)):          0.9957
Specificity (TN / (TN+FP)):          0.0739
Accuracy    ((TP+TN) / total):       0.4985
Predicted transfer (Spatial Criteria (Distance Threshold))   False    True 
Actual transfer                                                            
False                                                       192402  2412891
True                                                          9459  2215259


In [50]:
print_metrics(df_val, 'short_round_trip_flag', 'Spatial Criteria (Round Trip)')


=== Spatial Criteria (Round Trip) ===
TP: 2,221,450  TN: 9,433  FP: 2,595,860  FN: 3,268
Split rate  (FN / actual transfers): 0.0015
Merge rate  (FP / actual transfers): 1.1668
Sensitivity (TP / (TP+FN)):          0.9985
Specificity (TN / (TN+FP)):          0.0036
Accuracy    ((TP+TN) / total):       0.4619
Predicted transfer (Spatial Criteria (Round Trip))  False    True 
Actual transfer                                                   
False                                                9433  2595860
True                                                 3268  2221450


In [51]:
print_metrics(df_val, 'spatial_break_flag_final', 'Spatial Criteria (Combined)')


=== Spatial Criteria (Combined) ===
TP: 1,815,923  TN: 952,185  FP: 1,653,108  FN: 408,795
Split rate  (FN / actual transfers): 0.1838
Merge rate  (FP / actual transfers): 0.7431
Sensitivity (TP / (TP+FN)):          0.8162
Specificity (TN / (TN+FP)):          0.3655
Accuracy    ((TP+TN) / total):       0.5731
Predicted transfer (Spatial Criteria (Combined))   False    True 
Actual transfer                                                  
False                                             952185  1653108
True                                              408795  1815923


In [52]:
print_metrics(df_val, 'final_termination_flag', 'Binary + Temporal')


=== Binary + Temporal ===
TP: 1,628,699  TN: 2,543,844  FP: 61,449  FN: 596,019
Split rate  (FN / actual transfers): 0.2679
Merge rate  (FP / actual transfers): 0.0276
Sensitivity (TP / (TP+FN)):          0.7321
Specificity (TN / (TN+FP)):          0.9764
Accuracy    ((TP+TN) / total):       0.8639
Predicted transfer (Binary + Temporal)    False    True 
Actual transfer                                         
False                                   2543844    61449
True                                     596019  1628699


In [53]:
print_metrics(df_val, 'final_termination_flag_spatial', 'Binary + Temporal + Spatial (With Distance Threshold)')


=== Binary + Temporal + Spatial (With Distance Threshold) ===
TP: 1,617,002  TN: 2,557,538  FP: 47,755  FN: 607,716
Split rate  (FN / actual transfers): 0.2732
Merge rate  (FP / actual transfers): 0.0215
Sensitivity (TP / (TP+FN)):          0.7268
Specificity (TN / (TN+FP)):          0.9817
Accuracy    ((TP+TN) / total):       0.8643
Predicted transfer (Binary + Temporal + Spatial (With Distance Threshold))    False  \
Actual transfer                                                                       
False                                                                       2557538   
True                                                                         607716   

Predicted transfer (Binary + Temporal + Spatial (With Distance Threshold))    True   
Actual transfer                                                                      
False                                                                         47755  
True                                                    

In [54]:
print_metrics(df_val, 'final_termination_flag_spatial_nodist', 'Binary + Temporal + Spatial (Round Trip Only)')


=== Binary + Temporal + Spatial (Round Trip Only) ===
TP: 1,625,735  TN: 2,545,225  FP: 60,068  FN: 598,983
Split rate  (FN / actual transfers): 0.2692
Merge rate  (FP / actual transfers): 0.0270
Sensitivity (TP / (TP+FN)):          0.7308
Specificity (TN / (TN+FP)):          0.9769
Accuracy    ((TP+TN) / total):       0.8636
Predicted transfer (Binary + Temporal + Spatial (Round Trip Only))    False  \
Actual transfer                                                               
False                                                               2545225   
True                                                                 598983   

Predicted transfer (Binary + Temporal + Spatial (Round Trip Only))    True   
Actual transfer                                                              
False                                                                 60068  
True                                                                1625735  


In [55]:
temporal_errors = df_val[
    df_val['exceeds_time_allowance'] != (~df_val['true_transfer'])
].copy()

print(f"\nRows where temporal criterion disagrees with benchmark: {len(temporal_errors):,}")

print("\nTemporal errors by mode pair:")
print(temporal_errors['mode_pair'].value_counts(dropna=False))

print("\nTemporal errors by peak period:")
print(temporal_errors['peak_period'].value_counts(dropna=False))

print("\nTemporal errors by waiting time allowed:")
print(temporal_errors['waiting_time_allowed'].value_counts(dropna=False))

print("\nTemporal errors by reason:")
print(temporal_errors['temporal_flag_reason'].value_counts(dropna=False))


Rows where temporal criterion disagrees with benchmark: 390,719

Temporal errors by mode pair:
mode_pair
bus_bus        173665
train_bus      111830
bus_train       64670
train_train     39914
other             640
Name: count, dtype: int64

Temporal errors by peak period:
peak_period
off_peak        207456
evening_peak    126945
morning_peak     56318
Name: count, dtype: int64

Temporal errors by waiting time allowed:
waiting_time_allowed
12.0    202777
18.0    144767
10.0     42535
NaN        640
Name: count, dtype: int64

Temporal errors by reason:
temporal_flag_reason
exceeds_total_allowance        258058
within_total_allowance         129627
null_predicted_walking_time      2394
null_time_gap                     640
Name: count, dtype: int64


# journey dataset that imposes all criterias

In [56]:
gc.collect()

0

In [57]:
df4 = df4.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

# define final journey sequence:
# a row with final_termination_flag_spatial = True ends the current journey
df4['final_journey_seq'] = (
    df4.groupby('CRD_NUM')['final_termination_flag_spatial']
       .shift(fill_value=False)
       .groupby(df4['CRD_NUM'])
       .cumsum()
)

# first row of each final journey
journey_first = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_journey_seq'], keep='first')
       [[
           'CRD_NUM',
           'final_journey_seq',
           'ORIG_STATION_NAME',
           'ORIG_LOC_ID_NUM',
           'ORIG_MRK_ID_NUM',
           'ORIG_LATITUDE',
           'ORIG_LONGITUDE',
           'ENTRY_TM',
           'TRNSPT_MODE_CD',
           'PATRON_CATG_ID_NUM'
       ]]
       .rename(columns={
           'ORIG_STATION_NAME': 'journey_orig_station',
           'ORIG_LOC_ID_NUM': 'journey_orig_loc_id',
           'ORIG_MRK_ID_NUM': 'journey_orig_mrk_id',
           'ORIG_LATITUDE': 'journey_orig_latitude',
           'ORIG_LONGITUDE': 'journey_orig_longitude',
           'ENTRY_TM': 'journey_entry_tm',
           'TRNSPT_MODE_CD': 'journey_first_mode',
           'PATRON_CATG_ID_NUM': 'journey_patron_catg'
       })
)

# last row of each final journey
journey_last = (
    df4.drop_duplicates(subset=['CRD_NUM', 'final_journey_seq'], keep='last')
       [[
           'CRD_NUM',
           'final_journey_seq',
           'DEST_STATION_NAME',
           'DEST_LOC_ID_NUM',
           'DEST_MRK_ID_NUM',
           'DEST_LATITUDE',
           'DEST_LONGITUDE',
           'EXIT_TM',
           'TRNSPT_MODE_CD',
           'walk_distance',
           'next_orig_station',
           'final_termination_reason_spatial'
       ]]
       .rename(columns={
           'DEST_STATION_NAME': 'journey_dest_station',
           'DEST_LOC_ID_NUM': 'journey_dest_loc_id',
           'DEST_MRK_ID_NUM': 'journey_dest_mrk_id',
           'DEST_LATITUDE': 'journey_dest_latitude',
           'DEST_LONGITUDE': 'journey_dest_longitude',
           'EXIT_TM': 'journey_exit_tm',
           'TRNSPT_MODE_CD': 'journey_last_mode',
           'walk_distance': 'walk_to_next_journey_distance',
           'next_orig_station': 'next_journey_orig_station',
           'final_termination_reason_spatial': 'journey_end_reason'
       })
)

# aggregates within each final journey
journey_summary = (
    df4.groupby(['CRD_NUM', 'final_journey_seq'], as_index=False)
       .agg(
           num_stages=('ENTRY_TM', 'size'),
           total_ride_fare=('RIDE_FARE_AMT', 'sum'),
           total_ride_distance_km=('RIDE_DIST_KM_CNT', 'sum'),
           total_ride_time_mins=('RIDE_MIN_CNT', 'sum')
       )
)

# one row per final journey
df_journey = (
    journey_first
    .merge(journey_last, on=['CRD_NUM', 'final_journey_seq'], how='inner')
    .merge(journey_summary, on=['CRD_NUM', 'final_journey_seq'], how='left')
    .drop_duplicates(subset=['CRD_NUM', 'final_journey_seq'])
    .reset_index(drop=True)
)

# total door-to-door duration of the inferred journey
df_journey['journey_duration_mins'] = (
    df_journey['journey_exit_tm'] - df_journey['journey_entry_tm']
).dt.total_seconds() / 60

print(df_journey.shape)
print(df_journey[['CRD_NUM', 'final_journey_seq']].duplicated().sum(), "duplicate journey keys")

(6026404, 25)
0 duplicate journey keys


In [58]:
gc.collect()

0

In [59]:
# %who
# del df3, df5, df_val

In [60]:
gc.collect()

0

In [61]:
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

# LOAD SAVED PLANNING AREA JSON
with open('../../data/planning_areas_2019.json', 'r') as f:
    planning_area_json = json.load(f)

areas = pd.DataFrame(planning_area_json['SearchResults']).copy()
areas['geojson_parsed'] = areas['geojson'].apply(json.loads)
areas['geometry'] = areas['geojson_parsed'].apply(shape)

planning_area_gdf = gpd.GeoDataFrame(
    areas,
    geometry='geometry',
    crs='EPSG:4326'
)[['pln_area_n', 'geometry']].rename(columns={
    'pln_area_n': 'planning_area'
})


# HELPER FUNCTION TO ASSIGN PLANNING AREA
def get_planning_area(df, lat_col, lon_col):
    gdf_points = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326'
    )

    joined = gpd.sjoin(
        gdf_points,
        planning_area_gdf,
        how='left',
        predicate='within'
    )

    return joined['planning_area'].values


# ADD REGION COLUMNS TO df4
df4['orig_region'] = get_planning_area(df4, 'ORIG_LATITUDE', 'ORIG_LONGITUDE')
df4['dest_region'] = get_planning_area(df4, 'DEST_LATITUDE', 'DEST_LONGITUDE')


# HOUSE AREA = FIRST ORIGIN OBSERVED FOR EACH CRD_NUM
first_tap = (
    df4.sort_values(['CRD_NUM', 'ENTRY_TM'])
       .drop_duplicates(subset=['CRD_NUM'], keep='first')
       .copy()
)

first_tap['house_area'] = get_planning_area(
    first_tap,
    'ORIG_LATITUDE',
    'ORIG_LONGITUDE'
)

df4 = df4.merge(
    first_tap[['CRD_NUM', 'house_area']],
    on='CRD_NUM',
    how='left'
)

In [62]:
# ADD REGION COLUMNS TO df_journey
df_journey['journey_orig_region'] = get_planning_area(
    df_journey,
    'journey_orig_latitude',
    'journey_orig_longitude'
)

df_journey['journey_dest_region'] = get_planning_area(
    df_journey,
    'journey_dest_latitude',
    'journey_dest_longitude'
)

# MERGE HOUSE AREA INTO df_journey
df_journey = df_journey.merge(
    first_tap[['CRD_NUM', 'house_area']],
    on='CRD_NUM',
    how='left'
)

In [63]:
df4.columns

Index(['BUS_SVC_NUM', 'CRD_NUM', 'DEST_LOC_ID_NUM', 'ENTRY_DT', 'ENTRY_TM',
       'EXIT_DT', 'EXIT_TM', 'JRNY_ID_NUM', 'ORIG_LOC_ID_NUM', 'RIDE_DISC_AMT',
       'RIDE_DIST_KM_CNT', 'RIDE_FARE_AMT', 'RIDE_ID_NUM', 'RIDE_MIN_CNT',
       'PATRON_CATG_ID_NUM', 'TRNSPT_MODE_CD', 'DEST_STATION_NAME',
       'DEST_MRK_ID_NUM', 'DEST_LATITUDE', 'DEST_LONGITUDE',
       'DEST_Travel_Type', 'ORIG_STATION_NAME', 'ORIG_MRK_ID_NUM',
       'ORIG_LATITUDE', 'ORIG_LONGITUDE', 'ORIG_Travel_Type', 'next_orig_lat',
       'next_orig_lon', 'next_orig_station', 'walk_distance',
       'PATRON_CATG_DESC_TXT', 'walking_speed_ms', 'service_day',
       'next_ENTRY_TM', 'next_ORIG_LOC_ID_NUM', 'next_BUS_SVC_NUM',
       'next_TRNSPT_MODE_CD_x', 'is_last_stage', 'missing_info',
       'same_bus_service', 'same_station_consecutive',
       'return_or_intermediate', 'journey_termination_flag',
       'journey_termination_reason', 'full_journey_seq',
       'orig_journey_station', 'entry_journey_tm', 'dest_jou

In [64]:
print("df4 orig_region NaNs:", df4['orig_region'].isna().sum())
print("df4 dest_region NaNs:", df4['dest_region'].isna().sum())
print("df4 house_area NaNs:", df4['house_area'].isna().sum())


print("df_journey journey_orig_region NaNs:", df_journey['journey_orig_region'].isna().sum())
print("df_journey journey_dest_region NaNs:", df_journey['journey_dest_region'].isna().sum())
print("df_journey house_area NaNs:", df_journey['house_area'].isna().sum())

df4 orig_region NaNs: 23958
df4 dest_region NaNs: 49563
df4 house_area NaNs: 58278
df_journey journey_orig_region NaNs: 23958
df_journey journey_dest_region NaNs: 49563
df_journey house_area NaNs: 39923


In [65]:
# everyday we pickling
df4.to_pickle('../data/df4_lenient_with_regions.pkl')
df_journey.to_pickle('../data/df_lenient_journey_with_regions.pkl')

print('Saved df4_lenient_with_regions.pkl')
print('Saved df_lenient_journey_with_regions.pkl')

Saved df4_lenient_with_regions.pkl
Saved df_lenient_journey_with_regions.pkl


# Dont run this, for checking/

In [66]:
'''rides_ours = df4[:300].copy()

rides_ours.to_csv('rides_ours.csv', index=False)

# keep only common commuters
common_crds = set(df_journey['CRD_NUM']).intersection(set(df5['CRD_NUM']))

ours = df_journey[df_journey['CRD_NUM'].isin(common_crds)].copy()
theirs = df5[df5['CRD_NUM'].isin(common_crds)].copy()

# sort so first 200 is meaningful
ours = ours.sort_values(['CRD_NUM', 'journey_entry_tm']).reset_index(drop=True)
theirs = theirs.sort_values(['CRD_NUM', 'JRNY_START_TM']).reset_index(drop=True)

# take first 200 rows
ours_200 = ours.head(200).copy()
theirs_200 = theirs.head(200).copy()

# export
ours_200.to_csv('ours_200.csv', index=False)
theirs_200.to_csv('theirs_200.csv', index=False)'''

"rides_ours = df4[:300].copy()\n\nrides_ours.to_csv('rides_ours.csv', index=False)\n\n# keep only common commuters\ncommon_crds = set(df_journey['CRD_NUM']).intersection(set(df5['CRD_NUM']))\n\nours = df_journey[df_journey['CRD_NUM'].isin(common_crds)].copy()\ntheirs = df5[df5['CRD_NUM'].isin(common_crds)].copy()\n\n# sort so first 200 is meaningful\nours = ours.sort_values(['CRD_NUM', 'journey_entry_tm']).reset_index(drop=True)\ntheirs = theirs.sort_values(['CRD_NUM', 'JRNY_START_TM']).reset_index(drop=True)\n\n# take first 200 rows\nours_200 = ours.head(200).copy()\ntheirs_200 = theirs.head(200).copy()\n\n# export\nours_200.to_csv('ours_200.csv', index=False)\ntheirs_200.to_csv('theirs_200.csv', index=False)"